In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## Image segmentation with SAM 3

This notebook demonstrates how to use SAM 3 for image segmentation with text or visual prompts. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "face")
- **Box prompts**: Using bounding boxes as exemplar visual prompts

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_image_predictor_example.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [2]:
import os

import matplotlib.pyplot as plt
import numpy as np

import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [3]:
import torch

# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# Build Model

In [4]:
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)

In [5]:
# image_path = f"{sam3_root}/assets/images/test_image.jpg"

# # test path from local disk
# image_path = "/home/g20/projects/Training_Dataset/IMG_1908.jpeg"
# CONFIDENCE_THRESHOLD = 0.35

# image = Image.open(image_path)
# width, height = image.size
# processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD)
# inference_state = processor.set_image(image)

In [6]:
import os
import glob
import numpy as np
from PIL import Image
# Make sure to import your Sam3Processor and model initialization here

# --- Configuration ---
INPUT_DIR = "/home/g20/projects/BARC_dataset/2D Images of Multi-Conductors/3-Conductor/Training_Dataset/"
OUTPUT_DIR = "/home/g20/projects/BARC_dataset/2D Images of Multi-Conductors/3-Conductor/Labeled_Dataset_SAM3/" # Change this to your preferred base output path
CONFIDENCE_THRESHOLD = 0.35

# Map prompts to folder names AND the RGB color you want them to be in the combined mask
PROMPT_MAP = {
    "white wire":  {"folder": "White",  "color": [255, 255, 255]},
    "blue wire":   {"folder": "Blue",   "color": [0, 0, 255]},
    "red wire":    {"folder": "Red",    "color": [255, 0, 0]},
    "green wire":  {"folder": "Green",  "color": [0, 255, 0]},
    "yellow wire": {"folder": "Yellow", "color": [255, 255, 0]}
}

def extract_2d_mask(mask_tensor):
    """
    Extracts and collapses the mask tensor into a flat 2D numpy array.
    Returns None if no masks exist.
    """
    if mask_tensor is None or len(mask_tensor) == 0:
        return None

    masks_np = mask_tensor.detach().cpu().numpy()

    # Collapse the N dimension
    if masks_np.ndim == 4:
        masks_np = masks_np.squeeze(1) 
    
    # Merge all individual segments into one 2D silhouette
    return np.max(masks_np, axis=0)

def process_directory():
    # 1. Create all subfolders
    for info in PROMPT_MAP.values():
        os.makedirs(os.path.join(OUTPUT_DIR, info["folder"]), exist_ok=True)
        
    os.makedirs(os.path.join(OUTPUT_DIR, "Combined_Mask"), exist_ok=True)
    # os.makedirs(os.path.join(OUTPUT_DIR, "Masked"), exist_ok=True)

    # 2. Get all images
    image_paths = glob.glob(os.path.join(INPUT_DIR, "*.jpeg"))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, "*.jpg")))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, "*.png")))

    if not image_paths:
        print(f"No images found in {INPUT_DIR}")
        return

    # Initialize processor
    processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD)

    # 3. Iterate over every image
    for image_path in image_paths:
        filename = os.path.basename(image_path)
        base_filename = os.path.splitext(filename)[0]
        print(f"\nProcessing: {filename}")
        
        # Load image
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        inference_state = processor.set_image(image)
        
        # Create a blank black canvas for the color-coded combined mask
        combined_rgb_mask = np.zeros((height, width, 3), dtype=np.uint8)
        found_any_masks = False
        
        # 4. Iterate over every prompt
        for prompt, info in PROMPT_MAP.items():
            folder_name = info["folder"]
            rgb_color = info["color"]
            
            # Apply prompt
            processor.reset_all_prompts(inference_state)
            inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
            
            # Extract 2D binary mask
            raw_masks = inference_state.get("masks")
            binary_mask_2d = extract_2d_mask(raw_masks)
            
            if binary_mask_2d is not None:
                found_any_masks = True
                
                # --- A. Save the individual black & white mask ---
                mask_img_array = (binary_mask_2d * 255).astype(np.uint8)
                save_name = f"{base_filename}_{folder_name}_mask.png"
                save_path = os.path.join(OUTPUT_DIR, folder_name, save_name)
                Image.fromarray(mask_img_array).save(save_path)
                print(f"  ✓ Saved {folder_name} mask")
                
                # --- B. Add to the color-coded combined mask ---
                # Where the binary mask is 1, apply the assigned RGB color
                for channel in range(3):
                    combined_rgb_mask[:, :, channel] = np.where(
                        binary_mask_2d > 0, 
                        rgb_color[channel], 
                        combined_rgb_mask[:, :, channel]
                    )
            else:
                print(f"  ✗ No mask detected for '{prompt}'")

        # 5. Save the combined mask if at least one wire was found
        if found_any_masks:
            combined_save_name = f"{base_filename}_Combined_mask.png"
            combined_save_path = os.path.join(OUTPUT_DIR, "Combined_Mask", combined_save_name)
            Image.fromarray(combined_rgb_mask).save(combined_save_path)
            print(f"  ★ Saved Combined mask -> {combined_save_path}")

if __name__ == "__main__":
    process_directory()


Processing: IMG_1998.jpeg
  ✓ Saved White mask
  ✓ Saved Blue mask
  ✓ Saved Red mask
  ✗ No mask detected for 'green wire'
  ✗ No mask detected for 'yellow wire'
  ★ Saved Combined mask -> /home/g20/projects/BARC_dataset/2D Images of Multi-Conductors/3-Conductor/Labeled_Dataset_SAM3/Combined_Mask/IMG_1998_Combined_mask.png

Processing: IMG_1997.jpeg
  ✓ Saved White mask
  ✓ Saved Blue mask
  ✓ Saved Red mask
  ✗ No mask detected for 'green wire'
  ✗ No mask detected for 'yellow wire'
  ★ Saved Combined mask -> /home/g20/projects/BARC_dataset/2D Images of Multi-Conductors/3-Conductor/Labeled_Dataset_SAM3/Combined_Mask/IMG_1997_Combined_mask.png

Processing: IMG_2007.jpeg
  ✓ Saved White mask
  ✓ Saved Blue mask
  ✓ Saved Red mask
  ✗ No mask detected for 'green wire'
  ✗ No mask detected for 'yellow wire'
  ★ Saved Combined mask -> /home/g20/projects/BARC_dataset/2D Images of Multi-Conductors/3-Conductor/Labeled_Dataset_SAM3/Combined_Mask/IMG_2007_Combined_mask.png

Processing: IMG_19

# Text prompt

In [7]:
# def plot_binary_mask(result, ax=None):
#     mask = result["masks"][0].squeeze(0).cpu()  # Get the first mask, remove batch dim, move to CPU
#     # 1. Create a new figure ONLY if an axis wasn't provided
#     if ax is None:
#         plt.figure(figsize=(12, 8))
#         ax = plt.gca()
    
#     # 2. Ensure mask is on CPU/Numpy and squeezed to 2D
#     # Using .float() ensures the values are 0.0 and 1.0
#     mask_np = mask.squeeze().cpu().numpy().astype(np.float32)
    
#     # 3. Plot using a grayscale colormap
#     # vmin/vmax ensure 0 is always black and 1 is always white
#     ax.imshow(mask_np, cmap='gray', vmin=0, vmax=1)
    
#     # ax.axis('off') # Hide axes for a clean look

def plot_combined_binary_mask(mask_tensor, ax=None):
    """
    Args:
        mask_tensor: A tensor likely of shape [N, H, W] or [N, 1, H, W]
    """
    # 1. Move to CPU and convert to numpy
    # We use .detach() to ensure it's off the computation graph
    masks_np = mask_tensor.detach().cpu().numpy()

    # 2. Collapse the N dimension (the number of objects)
    # If shape is [N, 1, H, W], squeeze the '1' out
    if masks_np.ndim == 4:
        masks_np = masks_np.squeeze(1) 
    
    # Take the maximum across the object dimension (axis 0)
    # This merges all individual 'blue wire' segments into one silhouette
    combined_mask = np.max(masks_np, axis=0)

    # 3. Figure Setup
    if ax is None:
        plt.figure(figsize=(12, 8))
        ax = plt.gca()

    # 4. Plot Black and White
    # cmap='gray' makes 0=black, 1=white
    ax.imshow(combined_mask, cmap='gray', vmin=0, vmax=1)
    
    num_objects = masks_np.shape[0] if masks_np.ndim > 2 else 1
    ax.set_title(f"Combined Binary Mask | Objects detected: {num_objects}")
    ax.axis('off')
    
    plt.show()